# Study similarity - components A-E

## Where entity counts come from

Raw mentions are extracted from protocol text, linked to sections, then **standardized** to canonical entity ids. Similarity and recommendations reason over those canonical ids and their counts per study - the placeholder pipeline documents the intended graph operations without calling an LLM or database.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
from src import (
    entity_type_similarity,
    section_distribution_similarity,
    entity_deviation_level1,
    entity_deviation_level2,
    entity_deviation_combined,
    protocol_embedding_similarity,
    metadata_similarity,
)
from src.engine import StudySimilarityEngine, StudySimilarityInput

## A - Entity-type (mix)

Turn each study into **proportions** over canonical entities (composition, not scale), then **cosine** similarity.

In [ ]:
entity_keys = ["criterion_age", "criterion_liver", "endpoint_safety"]
a = {"criterion_age": 1, "criterion_liver": 1, "endpoint_safety": 2}
b = {"criterion_age": 1, "criterion_liver": 0, "endpoint_safety": 3}
s_a, art = entity_type_similarity(a, b, entity_keys)
s_a, art[:2]

## B - Section distribution

After you align sections across protocols, each column is total entity mentions in that **mapped** section. Row-normalize and take cosine.

In [ ]:
mapping = [("Intro", "Introduction"), ("Objectives", "Endpoints"), ("Excl", "Exclusions")]
sec_a = np.array([12.0, 50.0, 18.0])
sec_b = np.array([25.0, 48.0, 20.0])
section_distribution_similarity(sec_a, sec_b)

## C - Entity deviation (two levels)

**Level 1**: study-wide counts vs population mean mu - similar **magnitude** of deviation. **Level 2**: for salient entities, compare **sign** of section-level deviation vs the same mu pattern. Blended with 0.7/0.3 by default.

In [ ]:
population = [
    {"criterion_age": 2, "endpoint_safety": 5},
    {"criterion_age": 1, "endpoint_safety": 4},
    {"criterion_age": 3, "endpoint_safety": 6},
]
entity_keys2 = ["criterion_age", "endpoint_safety"]
ca = {"criterion_age": 4, "endpoint_safety": 8}
cb = {"criterion_age": 3, "endpoint_safety": 7}
l1, rows1 = entity_deviation_level1(ca, cb, population, entity_keys2)

def sec_vec(tot, hi, lo):
    return np.array([lo, hi, tot - hi - lo], dtype=float)

pop_sec = [
    {"criterion_age": sec_vec(4, 2, 1), "endpoint_safety": sec_vec(8, 5, 2)},
    {"criterion_age": sec_vec(3, 1, 1), "endpoint_safety": sec_vec(7, 4, 2)},
]
esa = {"criterion_age": sec_vec(4, 3, 0), "endpoint_safety": sec_vec(8, 6, 1)}
esb = {"criterion_age": sec_vec(3, 2, 0), "endpoint_safety": sec_vec(7, 5, 1)}
l2, rows2 = entity_deviation_level2(
    ca, cb, esa, esb, pop_sec, entity_keys2, step_threshold=0.5
)
entity_deviation_combined(l1, l2), l1, l2

## D - Non-entity protocol text

Chunks with **no** linked entities: L2-normalize each embedding, **mean** pool, optional unit renormalize, then cosine.

In [ ]:
chunks_a = [np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0])]
chunks_b = [np.array([1.0, 1.0, 0.0])]
protocol_embedding_similarity(chunks_a, chunks_b)

## E - Metadata

Per field: both missing -> skip; one missing -> mismatch; else compare (numeric or case-insensitive string).

In [ ]:
meta_a = {"phase": "Phase 2", "indication": "Alpha"}
meta_b = {"phase": "Phase 2", "indication": "Beta"}
metadata_similarity(meta_a, meta_b, ("phase", "indication", "design"))

## Full engine - weighted sum + artifacts

Default weights: entity-type 0.30, section 0.25, deviation 0.25, protocol embedding 0.15, metadata 0.05.

In [ ]:
keys = ["e1", "e2", "e3"]
pop_ent = [{"e1": 2, "e2": 3, "e3": 1}, {"e1": 1, "e2": 4, "e3": 2}]
pop_sec_full = [
    {k: np.array([1.0, 1.0, 0.0]) for k in keys},
    {k: np.array([1.0, 0.0, 1.0]) for k in keys},
]
inp = StudySimilarityInput(
    study_id_a="S-A",
    study_id_b="S-B",
    entity_counts_a={"e1": 3, "e2": 2, "e3": 1},
    entity_counts_b={"e1": 2, "e2": 3, "e3": 0},
    entity_keys=keys,
    section_totals_a=np.array([5.0, 4.0, 2.0]),
    section_totals_b=np.array([4.0, 5.0, 2.0]),
    section_mapping=[("A1", "B1"), ("A2", "B2"), ("A3", "B3")],
    population_entity_totals=pop_ent,
    entity_by_section_a={k: np.array([2.0, 1.0, 0.0]) for k in keys},
    entity_by_section_b={k: np.array([1.0, 2.0, 0.0]) for k in keys},
    population_section_per_entity=pop_sec_full,
    chunk_embeddings_a=[np.array([1.0, 0.0]), np.array([0.0, 1.0])],
    chunk_embeddings_b=[np.array([0.7, 0.7])],
    metadata_a={"phase": "P2", "indication": "X"},
    metadata_b={"phase": "P2", "indication": "X"},
)
eng = StudySimilarityEngine()
res = eng.compute(inp)
res.study_design_similarity, res.component_breakdown